# Null Handling Practice — `dropna` vs `fillna` vs `when/otherwise`

Part 2 of the interview drill on the dirty retail-orders dataset. Builds directly on **01 Deduplication Practice** in this folder: `dirty_orders` (24 raw rows) → `orders_dedup` (one row per `order_id`, 20 rows) → `orders_clean` (this notebook).

**The question:** `quantity` is a STRING column containing NULLs, negatives (−5, −2) and a padded `' 2 '`. Produce a clean **INT** column `quantity_clean` where NULL → 1 and negative → 1, valid values unchanged — built on the deduplicated table, not the raw one.

Full dirty-data menu lives in `demo_notebook` (Cell 4).

In [0]:
dirty_rows = [
    ("ORD1001", "John Smith", "USA", "Laptop", "2", "$1200.00", "2024-05-13", "2024-06-02 08:15:00"),
    ("ORD1002", " jane doe ", " canada", "Mouse", "5", "$25.50", "2024-05-14", "2024-06-02 08:15:00"),
    ("ORD1002", " jane doe ", " canada", "Mouse", "5", "$25.50", "2024-05-14", "2024-06-02 08:15:00"),
    ("ORD1003", None, "USA", "Keyboard", "1", "$45.00", "2024-05-15", "2024-06-02 08:15:00"),
    ("ORD1004", "Alice Johnson", "UK", "Monitor", "-5", "$180.00", "2024-05-16", "2024-06-02 08:15:00"),
    ("ORD1005", "bob brown", "usa", "Laptop", "3", None, "2024-05-17", "2024-06-02 08:15:00"),
    ("ORD1006", "Charlie Davis", "Canada", "Headphones", "0", "$60.00", "2024-05-18", "2024-06-02 08:15:00"),
    ("ORD1007", None, "USA", "Mouse", "2", "$25.50", None, "2024-06-02 08:15:00"),
    ("ORD1007", None, "USA", "Mouse", "9", "$25.50", "2024-05-22", "2024-06-03 10:05:00"),
    ("ORD1008", "David Lee", " USA ", "Keyboard", "1", "$45.00", "2024-05-20", "2024-06-02 08:15:00"),
    ("ORD1009", "Eve Wilson", "usa", "Monitor", "4", "$180.00", "2024-05-21", "2024-06-02 08:15:00"),
    ("ORD1009", "Eve Wilson", "usa", "Monitor", "2", "$180.00", "2024-06-10", "2024-06-03 09:40:00"),
    ("ORD1010", "Frank Miller", "Canada", "Laptop", "1", "$1200.00", "2050-01-01", "2024-06-02 08:15:00"),
    ("ORD1011", " grace taylor ", "USA", "Mouse", None, "$25.50", "2024-05-23", "2024-06-02 08:15:00"),
    ("ORD1012", "Henry Adams", "UK", "Headphones", "2", "$0.00", "2024-05-24", "2024-06-02 08:15:00"),
    ("ORD1013", "Ivy Clark", "CANADA", "Monitor", "2", "$180.00", "2024-05-25", "2024-06-02 08:15:00"),
    ("ORD1014", None, "usa", "Keyboard", "3", "$45.00", "2024-05-26", "2024-06-02 08:15:00"),
    ("ORD1015", "Jack White", "USA", "Laptop", None, None, None, "2024-06-02 08:15:00"),
    ("ORD1016", "Karen Hall", " Canada", "Mouse", "5", "$25.50", "2024-05-28", "2024-06-02 08:15:00"),
    ("ORD1017", "Leo King", "usa", "Monitor", "-2", "$180.00", "2024-05-29", "2024-06-02 08:15:00"),
    ("ORD1018", "Mona Scott", "USA", "Headphones", "1", "$60.00", "2024-05-30", "2024-06-02 08:15:00"),
    ("ORD1018", "Mona Scott", "USA", "Headphones", "7", "$60.00", "2024-05-30", "2024-06-03 11:20:00"),
    ("ORD1019", "Nick Green", "UK", "Keyboard", " 2 ", "$45.00", "2024-05-31", "2024-06-02 08:15:00"),
    ("ORD1020", "Olivia Baker", "Canada", "Mouse", "3", "$25.50", "2024-06-01", "2024-06-02 08:15:00"),
]

dirty_orders = spark.createDataFrame(
    dirty_rows,
    "order_id STRING, customer_name STRING, country STRING, product STRING, quantity STRING, unit_price STRING, order_date STRING, ingested_at STRING",
)

display(dirty_orders)

In [0]:
from pyspark.sql import Window
from pyspark.sql.functions import col, row_number

orders_dedup = dirty_orders.withColumn(
    'rn', row_number().over(Window.partitionBy(col('order_id')).orderBy(col('order_date').desc_nulls_last(), col('ingested_at').desc()))
).filter(col('rn') == 1)

orders_dedup.display()

## 1. Null handling on `quantity` — notes

**Question:** on `orders_dedup`, add a new column `quantity_clean` (INT): NULL → 1, negative → 1, padded `' 2 '` → 2, valid values unchanged. Keep the raw `quantity` column.

**Decision rules — which tool, when:**

* `dropna` — row-level **deletion**. Use when a row is too incomplete to be trusted and the business accepts losing it (e.g. ORD1015: NULL quantity AND price AND date — no revenue can ever come from it). It's data loss, so ask first.
* `fillna` — unconditional blanket replacement for the whole column. Safe only when one constant is correct for every NULL (e.g. empty discount → 0). Could NOT solve this question — "negative → 1" depends on the value.
* `when/otherwise` — conditional, value-dependent replacement. NULLs are just one branch, and it's the only one of the three that can also transform non-null values.

**Why dedup BEFORE cleaning:** cleaning rows you're about to throw away is wasted work, and a transform like "negative → 1" can make conflicting duplicates look identical — destroying the evidence needed to pick the right survivor.

**Business risk of filling NULL quantity with 1:** fabricates demand — 1 implies a real order of one unit, so revenue (qty × price) and inventory forecasts count phantom units. NULL might mean a cancelled/incomplete order, not "one unit". Safer: keep NULL (SUM/AVG ignore NULLs anyway), fill 0 if business says "no items ordered", or flag for review — and document the choice.

In [0]:
from pyspark.sql.functions import lit, when, trim, col

orders_clean = orders_dedup.drop("rn").withColumn(
    "quantity_clean",
    when(
        trim(col("quantity")).isNull() | (trim(col("quantity")).cast("int") < 0),
        lit(1),
    ).otherwise(trim(col("quantity")).cast("int")),
)

orders_clean.display()

In [0]:
orders_clean.select("order_id", "quantity", "quantity_clean") \
    .filter(col("order_id").isin("ORD1011", "ORD1015", "ORD1004", "ORD1017", "ORD1019", "ORD1006")) \
    .display()

## Key Concepts Summary

| Concept | Rule / Distinction |
| --- | --- |
| `dropna` | deletes whole rows — only when the row can't be trusted and losing it is acceptable |
| `fillna` | blanket constant for every NULL — safe only if one value is right for all of them |
| `when/otherwise` | value-dependent logic — the only one that handles "negative → 1" and transforms non-nulls too |
| New column vs overwrite | `withColumn("quantity_clean", ...)` keeps the raw `quantity` for audit — don't destroy source values |
| Pipeline order | dedup first, then standardize — cleaning before dedup wastes work and can mask conflicting duplicates |
| NULL → 1 risk | fabricates demand (revenue, inventory); alternatives: keep NULL, fill 0, or flag for review |

**Lessons from my attempts:**

* Two chained `withColumn`s on the same column, where the second runs `trim()` on an already-INT column, only works via Spark's implicit cast — it fails under ANSI SQL mode. One `when(isNull() | (cast < 0), 1).otherwise(cast)` does it safely.
* Drop the window helper column (`rn`) before a table moves to the next pipeline stage.

**Pipeline so far:** `dirty_orders` (24 rows) → `orders_dedup` (20 rows, Notebook 01) → `orders_clean` (20 rows, this notebook) → next: standardize `country` / clean `unit_price`.